# FFM_3_5 — 几何 → z：train / val / test 划分 + 按 epoch 训练 MLP（landmark 与 mesh 同一流程）

**为什么有这个 notebook**：之前的 landmark2z（FFM_2_9）和 mesh2z（FFM_3_2_1、3_3）都用交叉验证。这里换成更直观的
固定划分：训练集训练，**每个 epoch** 看验证集的曲线、按验证集 dB 选最好的 epoch，测试集**只评估一次**。

**划分（按人，两只耳永远在同一侧）**

| | landmark（SYMARE） | mesh（SONICOM） |
|---|---|---|
| test（锁定，只用一次） | AE 测试集 15 人 | AE 测试集 36 人（FFM_3_1 起锁定） |
| train / val | 其余 46 人按人随机 80 / 20 | 其余 164 人按人随机 80 / 20 |

测试集的人 AE 也没见过。**`RUN_TEST_*` 默认 False**；跑过一次后在 `OUT_DIR` 留下 `TEST_SPENT_<task>.json`，再跑会报错。

**同一划分上的对照**：① 人群均值 HRTF（训练耳）；② ridge（解析解，α 由验证集选）；③ 打乱标签训练的 MLP ——
它能显示「仅靠在验证集上挑 epoch」能凭空挑出多好的数。

**读数**：R²(z)、端到端 pooled dB、6–12 kHz、LSD、在 ④→① 空间里收回的比例、在本分区内认出本人的比例。

> 数据小，单次划分比交叉验证更抖：5 个随机种子取平均，并报种子间的离散。最终以 **test** 的数为准。

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, csv, zipfile, hashlib, time, math, copy, shutil, warnings
warnings.filterwarnings('ignore')
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
    os.system('pip install healpy -q')
    PATHS = dict(
        ckpt        = f'{SAVE_DIR}/FFM26_best.pth',
        lm_cache    = f'{SAVE_DIR}/landmark2z_v28_cache.npz',          # FFM_2_8 的导出
        symare      = f'{DRIVE}/ELEC5020/SYMARE.zip',
        mesh_latent = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',      # FFM_3_1 的导出
        samples     = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples',
        sonicom     = f'{DRIVE}/ELEC5020/SONICOM.zip',
    )
    OUT_DIR = f'{SAVE_DIR}/geometry2z_tvt_v2'      # v2：余弦退火 + 梯度裁剪 + 关 dropout 的训练损失

    # ---- 先把要反复读的东西复制到 Colab 本地盘，之后全部从本地读（每个 runtime 只复制一次）
    LOCAL = '/content/local_data'; os.makedirs(LOCAL, exist_ok=True)
    def stage_file(src):
        dst = os.path.join(LOCAL, os.path.basename(src))
        if not os.path.exists(dst):
            t0 = time.time(); shutil.copy2(src, dst + '.part'); os.replace(dst + '.part', dst)
            print(f'复制 {os.path.basename(src)} → 本地（{os.path.getsize(dst)/1e6:.0f} MB，{time.time()-t0:.0f} s）', flush=True)
        return dst
    def stage_samples(src_dir):
        dst = os.path.join(LOCAL, 'mesh2z_samples')
        if os.path.exists(os.path.join(dst, 'manifest.csv')):
            print('mesh2z_samples 已在本地', flush=True); return dst
        t0 = time.time(); z = src_dir.rstrip('/') + '.zip'
        if os.path.exists(z):                                  # 推荐：Drive 上放一个 zip，一次顺序读完
            print(f'从 {os.path.basename(z)} 解压网格样本 …', flush=True)
            with zipfile.ZipFile(z) as f: f.extractall(LOCAL)
        else:                                                  # 否则逐个文件复制（第一次较慢）
            files = sorted(os.listdir(src_dir)); os.makedirs(dst + '.part', exist_ok=True)
            for i, n in enumerate(files, 1):
                shutil.copy2(os.path.join(src_dir, n), os.path.join(dst + '.part', n))
                if i == 1 or i % 25 == 0 or i == len(files):
                    print(f'  复制网格样本 {i}/{len(files)}（{time.time()-t0:.0f} s）', flush=True)
            os.replace(dst + '.part', dst)
        assert os.path.exists(os.path.join(dst, 'manifest.csv')), f'{dst} 里没有 manifest.csv（zip 的顶层应是 mesh2z_samples/）'
        print(f'网格样本就绪：{len(os.listdir(dst))} 个文件，{time.time()-t0:.0f} s', flush=True); return dst
    for k in ('ckpt', 'lm_cache', 'mesh_latent'):
        if os.path.exists(PATHS[k]): PATHS[k] = stage_file(PATHS[k])
    if os.path.isdir(PATHS['samples']) or os.path.exists(PATHS['samples'].rstrip('/') + '.zip'):
        PATHS['samples'] = stage_samples(PATHS['samples'])
    # SONICOM.zip / SYMARE.zip 在 Cell 3 的 npy_dirs() 里解压到 /content/dataset（也是本地）
else:
    H = os.path.expanduser('~/Documents'); R = f'{H}/26S2/ELEC5021/hrtf-GitHub'
    PATHS = dict(
        ckpt        = os.environ.get('FFM26_CKPT', f'{R}/results/FFM26_best.pth'),
        lm_cache    = os.environ.get('LANDMARK_CACHE', f'{H}/26S2/ELEC5021/landmark2z_v28_cache.npz'),
        symare      = f'{H}/26S1/ELEC5020/SYMARE',
        mesh_latent = f'{R}/mesh2z/output/mesh2z_sonicom_latents_v1.npz',
        samples     = f'{H}/26S1/ELEC5020/SONICOM/mesh2z_samples',
        sonicom     = f'{H}/26S1/ELEC5020/SONICOM',
    )
    OUT_DIR = os.environ.get('GEOM2Z_OUT', f'{R}/mesh2z/output/geometry2z_tvt_v2')
os.makedirs(OUT_DIR, exist_ok=True)
for k, p in PATHS.items():
    print(f'{k:12s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先放好缺失的文件'

In [ ]:
# ==========================================================================
# Cell 1 — 配置（★ 跑之前定死）
# ==========================================================================
CFG = dict(
    TASKS      = ['landmark', 'mesh'],
    VAL_FRAC   = 0.20,           # 开发集里按人随机抽 20% 做 val
    SPLIT_SEED = 0,
    SEEDS      = [0],            # MLP 随机种子列表；多个时预测取平均。以后加种子：改成 [0, 1, 2]，已训练的种子直接从 checkpoint 载入
    K_PC       = 64,             # 目标 = z 在训练集上的前 K 个主成分
    landmark = dict(DID=3, HIDDEN=(128, 128), DROPOUT=0.1, LR=1e-3, WD=1e-2, BATCH=8,  EPOCHS=800,
                    RFF_M=32, RFF_SIGMA=0.02),          # FFM_2_9 最好配置：σ = 20 m⁻¹ = 0.02 mm⁻¹，M = 32
    mesh     = dict(DID=2, HIDDEN=(256, 128), DROPOUT=0.1, LR=1e-3, WD=1e-2, BATCH=16, EPOCHS=500,
                    SIGMAS=[0.005, 0.01, 0.02, 0.05, 0.1], RFF_M=256, Z_CUT=-75.0),   # FFM_3_2_1 最好配置
    ALPHA_GRID = list(np.logspace(-3, 6, 37)),
    NOTCH_HZ   = (6000, 12000),
    LR_MIN      = 1e-5,          # 余弦退火：每个模型的学习率从 LR 平滑降到 LR_MIN（与 AE 训练相同的调度）
    GRAD_CLIP   = 1.0,           # 梯度范数上限
    PRINT_EVERY = 10,            # 第 1 个、每 10 个、最后一个 epoch 打印一行
    SAVE_EVERY  = 10,            # 每 10 个 epoch 存一次 checkpoint（以及最后一个 epoch）
    RUN_TEST_LANDMARK = False,   # ★ 只在验证集上看完、决定不再改任何东西之后才改成 True，且只跑一次
    RUN_TEST_MESH     = False,
)
print(json.dumps({k: v for k, v in CFG.items() if k != 'ALPHA_GRID'}, indent=1))

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（与 FFM_2_3 / FFM_2_6 逐字相同，构建时复制进来）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 公共工具：AE 解码、HRTF 读取、指标、MLP、ridge、测试集保护
# ==========================================================================
import torch, torch.nn as nn, healpy as hp
import matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ck = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
N_BINS = ck['model_state_dict']['decoder.film.beta'].shape[1]
AE = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32, num_datasets=4,
                          num_bins=N_BINS, tie_encoder_film=True).to(device)
AE.load_state_dict(ck['model_state_dict']); AE.eval(); CK_SPLIT = ck['split']; del ck
FREQS = np.arange(N_BINS) * 44100 / 256
NOTCH = (FREQS >= CFG['NOTCH_HZ'][0]) & (FREQS <= CFG['NOTCH_HZ'][1])
_th, _ph = hp.pix2ang(16, np.arange(3072)); MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
print(f'AE 载入 | {N_BINS} bins | {device}')

def find_dir(root, target):
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp: continue
        if os.path.basename(dp) == target: c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True); return c[0][1] if c else None

def npy_dirs(src, name):
    if src.endswith('.zip'):
        out = f'/content/dataset/{name}'; os.makedirs(out, exist_ok=True)   # ★ 每个库单独解压
        if find_dir(out, 'npy_left') is None:
            t0 = time.time(); print(f'解压 {os.path.basename(src)} 到 {out} …（第一次要几分钟）', flush=True)
            with zipfile.ZipFile(src) as z: z.extractall(out)
            print(f'  解压完成，{time.time()-t0:.0f} s', flush=True)
        else:
            print(f'{name}: 已解压过，直接使用 {out}', flush=True)
        src = out
    d = (find_dir(src, 'npy_left'), find_dir(src, 'npy_right')); assert all(d), f'{name}: 找不到 npy_left/right'
    return d

def load_ear(dirs, sid, side):
    x = np.load(os.path.join(dirs[side], f"{sid}_{('left', 'right')[side]}_full.npy")).astype(np.float32)[:N_BINS]
    return x[:, MIRROR] if side == 1 else x

@torch.no_grad()
def decode(Zrows, did):
    out = np.zeros((len(Zrows), N_BINS, 3072), np.float32)
    for b in range(0, len(Zrows), 32):
        z = torch.from_numpy(Zrows[b:b + 32].reshape(-1, N_BINS, 64).astype(np.float32)).to(device)
        out[b:b + 32] = AE.decoder(z, torch.full((len(z),), did, device=device)).cpu().numpy()
    return out

def hrtf_metrics(P, X, side, base_db=None, rec_db=None):
    """P, X: (n_ears, 93, 3072). pooled = 所有耳 × bin × 方向（与 evaluate() 同口径）"""
    e2 = (P - X) ** 2; d = np.sqrt(e2.mean(1))
    out = dict(db=float(np.sqrt(e2.mean())), db_notch=float(np.sqrt(e2[:, NOTCH].mean())), lsd=float(d.mean()))
    ranks = []
    for s in (0, 1):                                  # 在本分区内、同侧耳里认人
        r = np.flatnonzero(side == s); a = P[r].reshape(len(r), -1); b = X[r].reshape(len(r), -1)
        d2 = (a * a).sum(1)[:, None] + (b * b).sum(1)[None] - 2 * a @ b.T
        ranks.append((d2 < d2.diagonal()[:, None]).sum(1) + 1)
    rk = np.concatenate(ranks); out['rank1'] = float(np.mean(rk == 1)); out['chance'] = 2 / len(P)
    if base_db is not None and rec_db is not None:
        out['recovered'] = (base_db - out['db']) / (base_db - rec_db)
    return out

def r2z(pred, Y, my):
    return float(1 - ((Y - pred) ** 2).sum() / ((Y - my) ** 2).sum())

class TargetPCA:
    """z -> 训练集上的前 K 个主成分 / scale；inverse 必须把 scale 乘回去（FFM_3_3 的教训）"""
    def __init__(self, Ytr, k):
        self.my = Ytr.mean(0); U, s, Vt = np.linalg.svd(Ytr - self.my, full_matrices=False)
        k = min(k, int((s > s[0] * 1e-9).sum())); self.V = Vt[:k].T
        T = (Ytr - self.my) @ self.V; self.scale = float(np.sqrt((T ** 2).sum(1).mean()))
    def forward(self, Y): return (Y - self.my) @ self.V / self.scale
    def inverse(self, T): return self.my + (T * self.scale) @ self.V.T

class MLP(nn.Module):
    def __init__(self, d_in, hidden, k, p):
        super().__init__(); L, d = [], d_in
        for h in hidden: L += [nn.Linear(d, h), nn.GELU(), nn.Dropout(p)]; d = h
        L.append(nn.Linear(d, k)); self.net = nn.Sequential(*L)
    def forward(self, x): return self.net(x)

def ridge_val(Xtr, Ytr, Xva, Yva, grid):
    """解析解 ridge；α 由验证集 z 空间误差选。返回 (预测函数, α)"""
    mx, sx = Xtr.mean(0), Xtr.std(0) + 1e-12; A = (Xtr - mx) / sx; my = Ytr.mean(0)
    U, s, Vt = np.linalg.svd(A, full_matrices=False); UtY = U.T @ (Ytr - my)
    sse = []
    for a in grid:
        W = Vt.T @ ((s / (s ** 2 + a))[:, None] * UtY)
        sse.append(((((Xva - mx) / sx) @ W + my - Yva) ** 2).sum())
    a = grid[int(np.argmin(sse))]; W = Vt.T @ ((s / (s ** 2 + a))[:, None] * UtY)
    return (lambda X: ((X - mx) / sx) @ W + my), float(a), int(np.argmin(sse))

def guard_test(task):
    mark = os.path.join(OUT_DIR, f'TEST_SPENT_{task}.json')
    assert not os.path.exists(mark), f'{task} 的测试集已经用过（{mark}）。不要再评估第二次。'
    return mark
print('工具就绪')

In [ ]:
# ==========================================================================
# Cell 4 — landmark 数据（SYMARE；FFM_2_8 的缓存：19 个点、左耳坐标系、FFM26 的 z）
# ==========================================================================
DATA, FEATURE_SPEC = {}, {}
if 'landmark' in CFG['TASKS']:
    C = np.load(PATHS['lm_cache'], allow_pickle=False)
    sids = [str(s) for s in C['sym']]; ae_part = [str(p) for p in C['ae_part']]
    xyz = C['xyz_ear'].astype(np.float64); Z = C['Z_ear'].astype(np.float64)
    n = len(sids); assert xyz.shape[0] == 2 * n and Z.shape[0] == 2 * n, '缓存的行应为 每人左、右 交替'
    assert str(C['coord_unit']) == 'mm' and Z.shape[1] == N_BINS * 64
    side = np.tile([0, 1], n)
    if 'side_is_right' in C.files: assert np.array_equal(C['side_is_right'].astype(int), side), '行序不是左右交替'
    # 每只耳的 19 个点减去自身质心（平移不变），单位 mm，不做缩放（大小是信号）
    q = xyz - xyz.mean(1, keepdims=True)
    lc = CFG['landmark']; G = np.random.default_rng(12345).standard_normal((lc['RFF_M'], 3))
    A = 2 * np.pi * lc['RFF_SIGMA'] * (q @ G.T)                     # (rows, 19, M) —— 每个点用同一个 B
    feats = np.concatenate([q.reshape(len(q), -1), np.cos(A).reshape(len(q), -1), np.sin(A).reshape(len(q), -1)], 1)
    FEATURE_SPEC['landmark'] = dict(points='19 landmarks per ear (FFM_2_8 ipsi set), left-ear frame',
        centre='per-ear landmark centroid', unit='mm', input='raw xyz (57) + per-landmark RFF',
        rff_M=lc['RFF_M'], rff_sigma_per_mm=lc['RFF_SIGMA'], G_seed=12345, dims=int(feats.shape[1]))
    DATA['landmark'] = dict(X=feats, Y=Z, sids=sids, part=ae_part, side=side, did=lc['DID'],
                            dirs=npy_dirs(PATHS['symare'], 'symare'))
    print(f"landmark：{n} 人 / {2*n} 只耳 | 输入 {feats.shape[1]} 维（原始 57 + RFF {feats.shape[1]-57}） | AE 分区 "
          f"{ {p: ae_part.count(p) for p in set(ae_part)} }")

In [ ]:
# ==========================================================================
# Cell 5 — mesh 数据（SONICOM；与 FFM_3_2_1 最好配置相同的特征：9 个简单量 + 耳廓多尺度 RFF）
# ==========================================================================
if 'mesh' in CFG['TASKS']:
    L = np.load(PATHS['mesh_latent'])
    dev = [str(s) for s in L['eval_sids']]; locked = [str(s) for s in L['locked_sids']]
    sids = dev + locked; part = ['dev'] * len(dev) + ['test'] * len(locked)
    IDX = {s: i for i, s in enumerate(L['sid'])}
    with open(os.path.join(PATHS['samples'], 'manifest.csv'), newline='') as f:
        MAN = {f"SON_{r['subject']}": r for r in csv.DictReader(f)}
    mc = CFG['mesh']; FLIP = np.array([1.0, -1.0, 1.0])
    G0 = torch.as_tensor(np.random.default_rng(12345).standard_normal((mc['RFF_M'], 3)).T, dtype=torch.float32, device=device)
    @torch.no_grad()
    def rff_all(P):
        base = 2 * math.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ G0)
        return np.concatenate([torch.cat([torch.cos(s * base).mean(0), torch.sin(s * base).mean(0)]).double().cpu().numpy()
                               for s in mc['SIGMAS']])
    pct = lambda a, x: float(np.percentile(a, x))
    feats, Y, side = [], [], []; t0 = time.time()
    for i_s, s in enumerate(sids, 1):
        if i_s == 1 or i_s % 25 == 0 or i_s == len(sids):
            print(f'  读取网格样本 {i_s}/{len(sids)}（{time.time()-t0:.0f} s）', flush=True)
        d = np.load(os.path.join(PATHS['samples'], f'{s[4:]}.npz'))
        head = d['head_pts'][:20000].astype(np.float64); head = head[head[:, 2] > mc['Z_CUT']]
        for sd, (pts, canal, area) in enumerate(((d['earL_pts'], d['ear_left_mm'].astype(float), float(MAN[s]['crop_area_L_mm2'])),
                                                 (d['earR_pts'], d['ear_right_mm'].astype(float) * FLIP, float(MAN[s]['crop_area_R_mm2'])))):
            hd = head if sd == 0 else head * FLIP
            q = pts[:20000].astype(np.float64) - canal; out = q[q[:, 1] > 3.0]
            simple = [canal[0], canal[1], canal[2], pct(q[:, 1], 95), pct(out[:, 2], 95) - pct(out[:, 2], 5),
                      pct(out[:, 0], 95) - pct(out[:, 0], 5), area, pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9)]
            feats.append(np.concatenate([simple, rff_all(q)])); Y.append(L['Z_ear'][IDX[s], sd].ravel()); side.append(sd)
    FEATURE_SPEC['mesh'] = dict(points='ear crop: 20k area-uniform points within 45 mm of the canal, canal-relative, right mirrored',
        input='9 simple measures + ear RFF mean embeddings', simple=['canal_x', 'canal_y', 'canal_z', 'pinna_protrusion',
        'pinna_height', 'pinna_length', 'crop_area', 'head_depth', 'head_height'], sigmas_per_mm=mc['SIGMAS'],
        rff_M=mc['RFF_M'], head_z_cut_mm=mc['Z_CUT'], G_seed=12345)
    DATA['mesh'] = dict(X=np.array(feats), Y=np.array(Y, np.float64), sids=sids, part=part, side=np.array(side),
                        did=mc['DID'], dirs=npy_dirs(PATHS['sonicom'], 'sonicom'))
    print(f"mesh：{len(sids)} 人（开发 {len(dev)}、锁定测试 {len(locked)}） | 输入 {DATA['mesh']['X'].shape[1]} 维")

In [ ]:
# ==========================================================================
# Cell 6 — 划分 · 基线 · ridge · MLP（每个 epoch 记录）—— 只用 train / val
# ==========================================================================
def make_split(D):
    n = len(D['sids']); dev = np.array([i for i in range(n) if D['part'][i] != 'test'])
    test = np.array([i for i in range(n) if D['part'][i] == 'test'])
    rng = np.random.default_rng(CFG['SPLIT_SEED']); perm = rng.permutation(dev)
    n_val = int(round(CFG['VAL_FRAC'] * len(dev))); val, tr = np.sort(perm[:n_val]), np.sort(perm[n_val:])
    rows = lambda subs: np.sort(np.concatenate([[2 * k, 2 * k + 1] for k in subs])).astype(int)
    S = dict(train=rows(tr), val=rows(val), test=rows(test), n_sub=dict(train=len(tr), val=len(val), test=len(test)))
    assert not (set(S['train']) & set(S['val'])) and not ((set(S['train']) | set(S['val'])) & set(S['test']))
    assert all(D['part'][k] == 'test' for k in test) and len(S['train']) + len(S['val']) + len(S['test']) == 2 * n
    return S

def load_partition(D, rows):
    return np.stack([load_ear(D['dirs'], D['sids'][r // 2], int(D['side'][r])) for r in rows])

def data_hash(D, S):
    h = hashlib.md5()
    for a in (D['X'], D['Y'], S['train'], S['val'], S['test']): h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

def atomic_save(obj, path):
    tmp = path + '.tmp'; torch.save(obj, tmp); os.replace(tmp, path)     # 断线也不会留下写了一半的文件

def train_mlp(task, D, S, pca, xs, mx, sx, cfg, seed, shuffled=False, Xva_hrtf=None):
    """训练一个 MLP；每 PRINT_EVERY 个 epoch 打印，每 SAVE_EVERY 个 epoch 存 checkpoint；有 checkpoint 就续训。"""
    tag = 'shuffled' if shuffled else f'seed{seed}'
    path = os.path.join(OUT_DIR, f'ckpt_{task}_{tag}.pth')
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr = torch.as_tensor(xs(D['X'][S['train']]), dtype=torch.float32, device=device)
    Xva = torch.as_tensor(xs(D['X'][S['val']]), dtype=torch.float32, device=device)
    Ytr = D['Y'][S['train']]
    if shuffled:                                       # 训练集内按人打乱（两只耳一起换），验证用真实目标
        subs = np.arange(len(Ytr) // 2); p = np.random.default_rng(1000 + seed).permutation(subs)
        Ytr = Ytr[np.array([2 * p[k] + s for k in subs for s in (0, 1)])]
    Ttr = torch.as_tensor(pca.forward(Ytr), dtype=torch.float32, device=device)
    model_args = dict(d_in=int(Xtr.shape[1]), hidden=tuple(cfg['HIDDEN']), k=int(Ttr.shape[1]), p=float(cfg['DROPOUT']))
    model = MLP(**model_args).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['LR'], weight_decay=cfg['WD'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['EPOCHS'], eta_min=CFG['LR_MIN'])
    my_tr = D['Y'][S['train']].mean(0); Yva = D['Y'][S['val']]
    meta = dict(task=task, tag=tag, seed=seed, shuffled=shuffled, data_hash=data_hash(D, S), model_args=model_args,
                cfg={k: v for k, v in cfg.items() if k != 'EPOCHS'}, k_pc=CFG['K_PC'], split_seed=CFG['SPLIT_SEED'],
                lr_min=CFG['LR_MIN'], grad_clip=CFG['GRAD_CLIP'], schedule='cosine',
                val_frac=CFG['VAL_FRAC'])
    hist = dict(train_loss=[], train_loss_eval=[], train_r2=[], val_loss=[], val_r2=[], val_db=[], lr=[])
    best = dict(db=np.inf, state=None, epoch=0); start = 1
    if os.path.exists(path):
        ck = torch.load(path, map_location=device, weights_only=False)
        assert ck['meta'] == meta, f'{path} 与当前数据/配置不一致 —— 删除它，或换一个 OUT_DIR'
        model.load_state_dict(ck['state_last']); opt.load_state_dict(ck['optimizer']); hist = ck['hist']
        sched.load_state_dict(ck['scheduler']); sched.T_max = cfg['EPOCHS']        # 改大 EPOCHS 续训时沿用同一条余弦
        best = dict(db=ck['best_val_db'], state=ck['state_best'], epoch=ck['best_epoch']); start = ck['epoch'] + 1
        torch.set_rng_state(ck['rng']['torch'].cpu())
        if torch.cuda.is_available() and ck['rng']['cuda'] is not None: torch.cuda.set_rng_state_all([r.cpu() for r in ck['rng']['cuda']])
        print(f"  [{task} {tag}] 载入 {os.path.basename(path)}：已完成 {ck['epoch']} 个 epoch"
              + ('，不需要再训练' if start > cfg['EPOCHS'] else f"，从 epoch {start} 续训到 {cfg['EPOCHS']}"))

    def save(ep):
        atomic_save(dict(
            meta=meta, epoch=ep, epochs_planned=cfg['EPOCHS'],
            state_last=model.state_dict(), state_best=best['state'], best_epoch=best['epoch'], best_val_db=best['db'],
            optimizer=opt.state_dict(), scheduler=sched.state_dict(), hist=hist,
            rng=dict(torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None),
            x_mean=mx, x_std=sx, pca=dict(my=pca.my, V=pca.V, scale=pca.scale),
            split={k: [D['sids'][r // 2] for r in S[k][::2]] for k in ('train', 'val', 'test')},
            feature_spec=FEATURE_SPEC[task], did=D['did'], ffm_ckpt=os.path.basename(PATHS['ckpt']),
            saved=time.strftime('%Y-%m-%d %H:%M:%S')), path)

    for ep in range(start, cfg['EPOCHS'] + 1):
        model.train(); order = torch.randperm(len(Xtr), device=device); tl = 0.0
        for b in range(0, len(Xtr), cfg['BATCH']):
            j = order[b:b + cfg['BATCH']]
            loss = ((model(Xtr[j]) - Ttr[j]) ** 2).sum(1).mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['GRAD_CLIP']); opt.step(); tl += loss.item() * len(j)
        lr_now = opt.param_groups[0]['lr']; sched.step()
        model.eval()
        with torch.no_grad():
            Ttr_hat = model(Xtr); Tva = model(Xva).cpu().numpy()
            tr_eval = float(((Ttr_hat - Ttr) ** 2).sum(1).mean())                       # 关 dropout、整个训练集
            tr_r2 = float(1 - ((Ttr_hat - Ttr) ** 2).sum() / ((Ttr - Ttr.mean(0)) ** 2).sum())
        Zva = pca.inverse(Tva)
        db = float(np.sqrt(((decode(Zva, D['did']) - Xva_hrtf) ** 2).mean()))
        hist['train_loss'].append(tl / len(Xtr)); hist['train_loss_eval'].append(tr_eval); hist['train_r2'].append(tr_r2)
        hist['val_loss'].append(float(((Tva - pca.forward(Yva)) ** 2).sum(1).mean()))
        hist['val_r2'].append(r2z(Zva, Yva, my_tr)); hist['val_db'].append(db); hist['lr'].append(lr_now)
        if db < best['db']: best = dict(db=db, state=copy.deepcopy(model.state_dict()), epoch=ep)
        if ep == 1 or ep % CFG['PRINT_EVERY'] == 0 or ep == cfg['EPOCHS']:
            print(f"  [{task} {tag}] ep {ep:4d}/{cfg['EPOCHS']} | lr {lr_now:.1e} | train {hist['train_loss'][-1]:.3f} "
                  f"(no dropout {tr_eval:.3f}, R² {tr_r2:+.3f}) | val loss {hist['val_loss'][-1]:.3f} | "
                  f"val R² {hist['val_r2'][-1]:+.4f} | val dB {db:.3f} | best {best['db']:.3f} @ {best['epoch']}")
        if ep % CFG['SAVE_EVERY'] == 0 or ep == cfg['EPOCHS']: save(ep)
    model.load_state_dict(best['state']); model.eval()
    return model, hist, best['epoch']

def model_from_ckpt(ck, which='best'):
    """单独从 .pth 恢复：模型 + 预测函数（输入为原始特征，输出为 z）"""
    m = MLP(**ck['meta']['model_args']).to(device)
    m.load_state_dict(ck['state_best' if which == 'best' else 'state_last']); m.eval()
    P = ck['pca']; mx, sx = ck['x_mean'], ck['x_std']
    def pred(X):
        with torch.no_grad():
            T = m(torch.as_tensor((X - mx) / sx, dtype=torch.float32, device=device)).cpu().numpy()
        return P['my'] + (T * P['scale']) @ P['V'].T
    return m, pred

def predict(model, pca, xs, X):
    with torch.no_grad():
        return pca.inverse(model(torch.as_tensor(xs(X), dtype=torch.float32, device=device)).cpu().numpy())

RUNS = {}
def run_task(task):
    D, cfg = DATA[task], CFG[task]; S = make_split(D)
    print(f"\n========== {task}：train {S['n_sub']['train']} / val {S['n_sub']['val']} / test {S['n_sub']['test']} 人（test 不读取）")
    # 输入标准化、目标 PCA：都只用训练集
    mx, sx = D['X'][S['train']].mean(0), D['X'][S['train']].std(0) + 1e-12
    xs = lambda X: (X - mx) / sx
    pca = TargetPCA(D['Y'][S['train']], CFG['K_PC'])
    # ★ 断言：完美输出必须还原出 z 在主成分子空间上的投影（防止漏乘 scale）
    proj = pca.my + (D['Y'][S['train']] - pca.my) @ pca.V @ pca.V.T
    assert np.allclose(pca.inverse(pca.forward(D['Y'][S['train']])), proj), 'TargetPCA.inverse 没有把 scale 乘回去'
    n_par = sum(p.numel() for p in MLP(D['X'].shape[1], cfg['HIDDEN'], pca.V.shape[1], cfg['DROPOUT']).parameters())
    print(f"MLP {D['X'].shape[1]} → {cfg['HIDDEN']} → {pca.V.shape[1]} 主成分 | {n_par:,} 个参数 | 训练耳 {len(S['train'])}")

    # 验证集真实 HRTF、训练耳均值、AE 重建
    Xva_h = load_partition(D, S['val']); mu = np.zeros((N_BINS, 3072))
    for r in S['train']: mu += load_ear(D['dirs'], D['sids'][r // 2], int(D['side'][r]))
    mu /= len(S['train']); side_va = D['side'][S['val']]; Yva = D['Y'][S['val']]; my_tr = D['Y'][S['train']].mean(0)
    base = hrtf_metrics(np.broadcast_to(mu, Xva_h.shape).astype(np.float32), Xva_h, side_va)
    rec = hrtf_metrics(decode(Yva, D['did']), Xva_h, side_va)
    ridge_f, alpha, ai = ridge_val(D['X'][S['train']], D['Y'][S['train']], D['X'][S['val']], Yva, CFG['ALPHA_GRID'])
    rows = {'population mean (train ears)': dict(r2=0.0, **base),
            f'ridge (alpha chosen on val)': dict(r2=r2z(ridge_f(D['X'][S['val']]), Yva, my_tr),
                                                  **hrtf_metrics(decode(ridge_f(D['X'][S['val']]), D['did']), Xva_h, side_va, base['db'], rec['db']))}
    if ai in (0, len(CFG['ALPHA_GRID']) - 1): print('★ ridge 的 α 选在网格边上')

    t0 = time.time(); models, hists, best_eps, preds = [], [], [], []
    for sd in CFG['SEEDS']:
        m, h, be = train_mlp(task, D, S, pca, xs, mx, sx, cfg, seed=sd, Xva_hrtf=Xva_h)
        models.append(m); hists.append(h); best_eps.append(be); preds.append(predict(m, pca, xs, D['X'][S['val']]))
        print(f"  seed {sd}: 最好 epoch {be:4d} | val dB {min(h['val_db']):.3f} | val R²(z) {h['val_r2'][be-1]:+.4f}  ({time.time()-t0:.0f} s)")
    ens = np.mean(preds, 0)
    rows[f'MLP, seeds {CFG["SEEDS"]} averaged'] = dict(r2=r2z(ens, Yva, my_tr),
        **hrtf_metrics(decode(ens, D['did']), Xva_h, side_va, base['db'], rec['db']))
    seed_db = [min(h['val_db']) for h in hists]
    ms, hs, bs = train_mlp(task, D, S, pca, xs, mx, sx, cfg, seed=99, shuffled=True, Xva_hrtf=Xva_h)
    ps = predict(ms, pca, xs, D['X'][S['val']])
    rows['MLP on shuffled targets (control)'] = dict(r2=r2z(ps, Yva, my_tr),
        **hrtf_metrics(decode(ps, D['did']), Xva_h, side_va, base['db'], rec['db']))
    rows['AE reconstruction (floor)'] = dict(r2=1.0, **rec)

    print(f"\n{task} — VALIDATION（{S['n_sub']['val']} 人 / {len(S['val'])} 只耳）")
    print(f"{'':36s}{'R²(z)':>8s}{'pooled':>8s}{'6–12k':>8s}{'LSD':>7s}{'收回':>8s}{'认人':>7s}")
    for k, r in rows.items():
        rv = f"{r['recovered']:8.1%}" if 'recovered' in r else f"{'':>8s}"
        print(f"{k:36s}{r['r2']:+8.4f}{r['db']:8.3f}{r['db_notch']:8.3f}{r['lsd']:7.3f}{rv}{r['rank1']:7.1%}")
    print(f"认人随机水平 {rows['population mean (train ears)']['chance']:.1%} | 各种子最好 val dB：{np.round(seed_db, 3)}"
          f"（离散 ±{np.std(seed_db):.3f}）| 打乱对照最好 epoch {bs}")

    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for sd, h in enumerate(hists):
        ax[0].plot(h['train_loss_eval'], c='C0', alpha=.6, lw=1); ax[0].plot(h['val_loss'], c='C1', alpha=.6, lw=1)
        ax[1].plot(h['val_r2'], c='C0', alpha=.6, lw=1); ax[2].plot(h['val_db'], c='C0', alpha=.6, lw=1)
    ax[1].plot(hs['val_r2'], c='C3', ls='--', lw=1, label='shuffled-target control'); ax[2].plot(hs['val_db'], c='C3', ls='--', lw=1, label='shuffled-target control')
    for y, c, lab in ((base['db'], 'k', 'population mean'), (rows['ridge (alpha chosen on val)']['db'], 'C2', 'ridge'), (rec['db'], 'grey', 'AE reconstruction')):
        ax[2].axhline(y, c=c, lw=1, ls=':', label=lab)
    ax[0].set_title(f'{task}: train, no dropout (blue) / val (orange) loss'); ax[0].set_yscale('log')
    ax[1].axhline(0, c='k', lw=.6); ax[1].set_title('val $R^2(z)$ per epoch'); ax[2].set_title('val pooled dB per epoch')
    for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
    ax[2].legend(fontsize=7); ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'{task}_curves.png'), dpi=150); plt.show()
    RUNS[task] = dict(S=S, xs=xs, pca=pca, models=models, ridge=ridge_f, mu=mu, val=rows, best_epochs=best_eps,
                      seed_val_db=seed_db, n_params=n_par, alpha=alpha)

In [ ]:
# ==========================================================================
# Cell 7 — 跑 landmark
# ==========================================================================
if 'landmark' in CFG['TASKS']: run_task('landmark')

In [ ]:
# ==========================================================================
# Cell 8 — 跑 mesh
# ==========================================================================
if 'mesh' in CFG['TASKS']: run_task('mesh')

In [ ]:
# ==========================================================================
# Cell 8b — 只读 .pth：画全部曲线、恢复模型（重启 runtime 后，跑完 Cell 0–3 即可用）
# ==========================================================================
import glob
def plot_from_ckpts(task):
    files = sorted(glob.glob(os.path.join(OUT_DIR, f'ckpt_{task}_*.pth')))
    assert files, f'{OUT_DIR} 里没有 {task} 的 checkpoint'
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    for f in files:
        ck = torch.load(f, map_location='cpu', weights_only=False); h = ck['hist']; tag = ck['meta']['tag']
        ls = '--' if ck['meta']['shuffled'] else '-'
        ax[0].plot(h.get('train_loss_eval', h['train_loss']), ls, lw=1, label=f'{tag} train'); ax[0].plot(h['val_loss'], ls, lw=1, alpha=.6)
        ax[1].plot(h['val_r2'], ls, lw=1, label=tag); ax[2].plot(h['val_db'], ls, lw=1, label=tag)
        print(f"{os.path.basename(f):28s} 完成 {ck['epoch']}/{ck['epochs_planned']} epoch | 最好 val dB {ck['best_val_db']:.3f} "
              f"@ {ck['best_epoch']} | 存于 {ck['saved']} | train/val/test {len(ck['split']['train'])}/{len(ck['split']['val'])}/{len(ck['split']['test'])} 人")
    ax[0].set_yscale('log'); ax[0].set_title(f'{task}: loss (solid train, faint val)')
    ax[1].set_title('val $R^2(z)$'); ax[2].set_title('val pooled dB')
    for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3); a.legend(fontsize=6)
    plt.tight_layout(); plt.show()

for task in CFG['TASKS']:
    if glob.glob(os.path.join(OUT_DIR, f'ckpt_{task}_*.pth')): plot_from_ckpts(task)
# 例：单独恢复一个模型做预测
# ck = torch.load(os.path.join(OUT_DIR, 'ckpt_mesh_seed0.pth'), map_location=device, weights_only=False)
# model, pred = model_from_ckpt(ck); z_hat = pred(DATA['mesh']['X'][:4])

In [ ]:
# ==========================================================================
# Cell 9 — ★ TEST：只在决定不再改任何东西之后运行，每个任务只运行一次
# ==========================================================================
TEST = {}
for task in CFG['TASKS']:
    if not CFG[f'RUN_TEST_{task.upper()}']:
        print(f'{task}: RUN_TEST_{task.upper()} = False —— 测试集未读取'); continue
    mark = guard_test(task); D, R = DATA[task], RUNS[task]; S = R['S']
    Xte_h = load_partition(D, S['test']); side_te = D['side'][S['test']]; Yte = D['Y'][S['test']]
    my_tr = D['Y'][S['train']].mean(0)
    base = hrtf_metrics(np.broadcast_to(R['mu'], Xte_h.shape).astype(np.float32), Xte_h, side_te)
    rec = hrtf_metrics(decode(Yte, D['did']), Xte_h, side_te)
    ens = np.mean([predict(m, R['pca'], R['xs'], D['X'][S['test']]) for m in R['models']], 0)
    rid = R['ridge'](D['X'][S['test']])
    rows = {'population mean (train ears)': dict(r2=0.0, **base),
            'ridge': dict(r2=r2z(rid, Yte, my_tr), **hrtf_metrics(decode(rid, D['did']), Xte_h, side_te, base['db'], rec['db'])),
            'MLP, seeds averaged': dict(r2=r2z(ens, Yte, my_tr), **hrtf_metrics(decode(ens, D['did']), Xte_h, side_te, base['db'], rec['db'])),
            'AE reconstruction (floor)': dict(r2=1.0, **rec)}
    print(f"\n{task} — TEST（{S['n_sub']['test']} 人，第一次也是唯一一次）")
    for k, r in rows.items():
        rv = f"{r['recovered']:8.1%}" if 'recovered' in r else f"{'':>8s}"
        print(f"{k:32s} R²(z) {r['r2']:+.4f} | pooled {r['db']:.3f} | 6–12k {r['db_notch']:.3f} | LSD {r['lsd']:.3f} | 收回 {rv} | 认人 {r['rank1']:.1%}")
    TEST[task] = rows
    json.dump(dict(time=time.strftime('%Y-%m-%d %H:%M:%S'), rows=rows), open(mark, 'w'), indent=1, default=float)
    print(f'已记录：{mark} —— 这个测试集现在已经用过了')

In [ ]:
# ==========================================================================
# Cell 10 — 导出
# ==========================================================================
out = dict(cfg={k: v for k, v in CFG.items() if k != 'ALPHA_GRID'},
           runs={t: dict(n_subjects=R['S']['n_sub'], n_params=R['n_params'], best_epochs=R['best_epochs'],
                         seed_val_db=R['seed_val_db'], ridge_alpha=R['alpha'], val=R['val']) for t, R in RUNS.items()},
           test=TEST)
json.dump(out, open(os.path.join(OUT_DIR, 'results.json'), 'w'), indent=1, ensure_ascii=False, default=float)
print(f'-> {OUT_DIR}/results.json, *_curves.png')

## 读结果之前
1. **按验证集挑 epoch 会让验证集的数偏乐观**：打乱标签的对照也会挑出一个「最好 epoch」——它的验证分数就是这种乐观有多大。
   MLP 的验证分数要和它比，而不是和 0 比。
2. 种子间的离散（Cell 7/8 最后一行）就是验证集的分辨率：小于它的差别不要解读。
3. **test 只跑一次**，跑之前确认 CFG 不会再改。跑完后的数才是最终数字。
4. landmark 的 val 只有约 9 人、mesh 约 33 人：这是单次划分的固有代价，也是为什么之前用交叉验证。两种方法的结论应当一致。